# Módulo 9: Grafos multi-agente y A2A (25 min)

En el Módulo 6 un agente orquestador decidía cuándo llamar a un especialista (*agents as tools*). En este módulo **tú diseñas el flujo**: con el patrón **Graph** de Strands defines qué agentes participan (nodos), en qué orden (aristas) y bajo qué condiciones. Al final conectarás un nodo que vive en **otro servicio** usando el protocolo **A2A (Agent-to-Agent)**.

Basado en la guía oficial [Graph](https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph/) y [Agent-to-Agent (A2A)](https://strandsagents.com/docs/user-guide/sdk/multi-agent/agent-to-agent/) de Strands Agents.

**Prerrequisitos:** Módulos 1 y 6. Credenciales de AWS configuradas (Strands usa Amazon Bedrock por defecto) y el extra A2A instalado: `uv sync` en la raíz del proyecto, o bien `pip install 'strands-agents[a2a]'`.

**Archivos de esta carpeta:**

| Archivo | Para qué sirve |
|---|---|
| `herramientas_soporte.py` | Herramienta y fábrica del especialista de soporte técnico (la usan el notebook y el servidor) |
| `servidor_soporte_tecnico.py` | Expone a ese especialista como servicio A2A |

---

## Contexto: ¿qué es un Graph?

Un **Graph** te da control determinista sobre cómo se ejecuta un conjunto de agentes:

1. Los **nodos** son agentes, agentes remotos (A2A) u otros sistemas multi-agente (otro Graph o un Swarm).
2. Las **aristas** definen las dependencias y por dónde fluye la información.
3. La ejecución sigue la estructura del grafo y respeta las dependencias.
4. La **salida de un nodo es la entrada** de los nodos que dependen de él.
5. Los **puntos de entrada** reciben la tarea original.
6. Se admiten **ciclos** (por ejemplo, bucles de revisión) siempre que tengan condiciones de salida y límites.

### ¿Cuándo usar Graph?

La gran diferencia entre los patrones multi-agente es **quién decide el camino**:

| Patrón | Quién decide el camino | Ejemplo típico |
|---|---|---|
| *Agents as tools* (Módulo 6) | Un agente orquestador elige a qué especialista llamar | Asistente que delega tareas puntuales |
| **Graph** | **Tú** defines nodos y aristas; las condiciones eligen la rama | Procesos de negocio con pasos, ramas y bucles |
| Swarm | Los propios agentes se pasan el control entre sí | Colaboración abierta y creativa |
| Workflow | Un DAG fijo de tareas, sin ciclos | Operaciones repetibles en paralelo |

La documentación pone como ejemplo de Graph el **soporte al cliente**: enrutar cada conversación según la intención del usuario. Eso es lo que vas a construir.

### El caso de hoy: tickets de soporte de TechStore

Vas a construir el sistema poco a poco, una idea por parte:

| Parte | Qué aprendes | Forma del grafo |
|---|---|---|
| 1 | Nodos, aristas y resultados | Diamante: un nodo, dos en paralelo y un agregador |
| 2 | Qué recibe cada nodo | (inspección del grafo anterior) |
| 3 | Aristas condicionales y contexto en tiempo de ejecución | Ramas según la intención del ticket |
| 4 | Ciclos con límites de seguridad | Bucle redactor ⇄ revisor |
| 5 | Streaming de eventos | (el grafo de la Parte 1, en directo) |
| 6 | **A2A**: un nodo que vive en otro servicio | Ramas con un nodo remoto |

---

## Parte 0: Preparación

Además de los tickets de ejemplo, definimos dos funciones para **visualizar cualquier grafo con Mermaid**:

- `a_mermaid(grafo)` recorre las aristas del objeto `Graph` real y genera el diagrama. Las aristas condicionales muestran el nombre de su función de condición y los nodos remotos (A2A) se pintan de otro color.
- `mermaid(diagrama)` lo dibuja con el servicio público [mermaid.ink](https://mermaid.ink) (requiere conexión a Internet).

Como el diagrama sale del grafo construido y no de un dibujo hecho a mano, **siempre coincide con lo que ejecutas**.

In [1]:
import base64

from strands import Agent
from strands.multiagent import GraphBuilder
from IPython.display import Image, Markdown, display

TICKET_TECNICO = (
    "Hola, compré unos Auriculares Inalámbricos (pedido ORD-5521) hace dos semanas y no se "
    "emparejan con mi teléfono. Ya los reinicié y nada. ¡Estoy bastante molesto!"
)
TICKET_FACTURACION = (
    "Buenas, me han cobrado dos veces el pedido ORD-7788 del Teclado Mecánico. "
    "Quiero que me devuelvan el cargo duplicado, por favor."
)

POLITICAS = """- Garantía: 1 año en todos los dispositivos. Los defectos de fábrica se reparan o sustituyen gratis.
- Devoluciones: 30 días desde la entrega, con el producto en buen estado.
- Cobros duplicados: se reembolsan en 3 a 5 días hábiles tras verificar el cargo."""


def resumen(resultado) -> None:
    """Muestra el estado, el orden de ejecución y las métricas de un grafo."""
    print(f"Estado: {resultado.status}")
    print(f"Orden de ejecución: {' → '.join(nodo.node_id for nodo in resultado.execution_order)}")
    print(f"Tiempo: {resultado.execution_time / 1000:.1f} s | Tokens: {resultado.accumulated_usage['totalTokens']}")


def a_mermaid(grafo) -> str:
    """Convierte un Graph de Strands en un diagrama Mermaid."""
    lineas = ["flowchart LR"]
    for arista in sorted(grafo.edges, key=lambda a: (a.from_node.node_id, a.to_node.node_id)):
        origen, destino = arista.from_node.node_id, arista.to_node.node_id
        if arista.condition:
            etiqueta = getattr(arista.condition, "__name__", "condición")
            lineas.append(f"    {origen} -->|{etiqueta}| {destino}")
        else:
            lineas.append(f"    {origen} --> {destino}")
    # Los nodos remotos (A2A) se pintan de otro color
    remotos = [nodo.node_id for nodo in grafo.nodes.values() if type(nodo.executor).__name__ == "A2AAgent"]
    if remotos:
        lineas.append("    classDef remoto fill:#fde68a,stroke:#b45309")
        lineas.append(f"    class {','.join(remotos)} remoto")
    return "\n".join(lineas)


def mermaid(diagrama: str) -> None:
    """Dibuja un diagrama Mermaid con el servicio mermaid.ink."""
    b64 = base64.b64encode(diagrama.encode("utf8")).decode("ascii")
    display(Image(url=f"https://mermaid.ink/img/{b64}"))


print("✅ Listo")

✅ Listo


---

## Parte 1: Tu primer grafo — trabajo en paralelo con agregación

```
                     ┌──────────────┐
                ┌──▶ │ sentimiento  │ ──┐
┌───────────┐   │    └──────────────┘   │    ┌───────────┐
│ recepcion │ ──┤                       ├──▶ │ redactor  │ ──▶ respuesta al cliente
└───────────┘   │    ┌──────────────┐   │    └───────────┘
                └──▶ │  politicas   │ ──┘
                     └──────────────┘
```

- `recepcion` (punto de entrada) extrae los datos clave del ticket.
- `sentimiento` y `politicas` no dependen entre sí, así que **se ejecutan en paralelo**.
- `redactor` combina ambos resultados en la respuesta final.

Con `GraphBuilder` añades nodos (`add_node(agente, "id")`), aristas (`add_edge("origen", "destino")`) y, opcionalmente, el punto de entrada (si no lo indicas, se detecta solo: son los nodos sin aristas de entrada).

> 💡 Construimos el grafo dentro de una **función**. Los agentes guardan su historial de conversación, así que si reutilizáramos el mismo grafo para otro ticket, sus agentes "recordarían" el anterior. Con una función obtenemos agentes nuevos en cada ejecución.

In [2]:
def construir_grafo_respuesta():
    recepcion = Agent(
        name="recepcion",
        system_prompt="Extrae del ticket de soporte el producto, el número de pedido y el problema. "
        "Responde con una lista de 3 líneas.",
        callback_handler=None,
    )
    sentimiento = Agent(
        name="sentimiento",
        system_prompt="Evalúa el tono del cliente (tranquilo, molesto o furioso) y la urgencia "
        "(baja, media o alta). Responde en 2 líneas.",
        callback_handler=None,
    )
    politicas = Agent(
        name="politicas",
        system_prompt=f"Indica qué política de la tienda aplica al caso y por qué, en 2 líneas.\n\nPolíticas:\n{POLITICAS}",
        callback_handler=None,
    )
    redactor = Agent(
        name="redactor",
        system_prompt="Redacta en español la respuesta final al cliente (máximo 120 palabras). "
        "Adapta el tono a su estado de ánimo y aplica la política indicada.",
        callback_handler=None,
    )

    builder = GraphBuilder()
    builder.add_node(recepcion, "recepcion")
    builder.add_node(sentimiento, "sentimiento")
    builder.add_node(politicas, "politicas")
    builder.add_node(redactor, "redactor")

    builder.add_edge("recepcion", "sentimiento")
    builder.add_edge("recepcion", "politicas")
    builder.add_edge("sentimiento", "redactor")
    builder.add_edge("politicas", "redactor")

    builder.set_entry_point("recepcion")
    builder.set_execution_timeout(300)  # límite de seguridad: 5 minutos
    return builder.build()


grafo = construir_grafo_respuesta()
mermaid(a_mermaid(grafo))

Este es el grafo que acabas de construir, dibujado a partir del propio objeto `Graph`. Ahora lo ejecutamos con un ticket:

In [3]:
resultado = grafo(TICKET_TECNICO)
resumen(resultado)

Estado: Status.COMPLETED
Orden de ejecución: recepcion → sentimiento → politicas → redactor
Tiempo: 6.7 s | Tokens: 1338


El resultado del grafo (`GraphResult`) incluye el estado, el orden de ejecución, métricas y **el resultado de cada nodo** en `resultado.results["id_del_nodo"]`:

In [4]:
for nodo in ["recepcion", "sentimiento", "politicas", "redactor"]:
    display(Markdown(f"### 🧩 `{nodo}`\n\n{resultado.results[nodo].result}"))

### 🧩 `recepcion`

Aquí está la información extraída del ticket:

- **Producto:** Auriculares Inalámbricos
- **Número de pedido:** ORD-5521
- **Problema:** Los auriculares no se emparejan con el teléfono, incluso después de haberlos reiniciado


### 🧩 `sentimiento`

**Tono:** Molesto — el cliente expresa frustración explícita ("¡Estoy bastante molesto!") pero sin llegar a ser agresivo o iracundo.
**Urgencia:** Media — el problema lleva dos semanas y ya intentó resolverlo solo, lo que aumenta la impaciencia, aunque no reporta una necesidad crítica inmediata.


### 🧩 `politicas`

**Política aplicable: Garantía** — Los auriculares presentan un fallo de funcionamiento (no emparejamiento) tras solo dos semanas de uso, lo que indica un posible defecto de fábrica cubierto por el año de garantía.


### 🧩 `redactor`

Hola, entiendo perfectamente tu frustración y lamentamos los inconvenientes con tu pedido ORD-5521. Dos semanas con este problema es demasiado tiempo, y tienes toda la razón en estar molesto.

Dado que los auriculares presentan este fallo tan pronto, está cubierto por nuestra **garantía de un año**. Vamos a solucionarlo de inmediato: iniciaremos el proceso de **reemplazo o reparación** sin coste alguno para ti.

En las próximas **24 horas** recibirás un correo con las instrucciones para la devolución y los detalles del envío del nuevo dispositivo.

¿Prefieres un reemplazo directo o una revisión técnica? Con gusto gestionamos lo que mejor te convenga. 🎧


---

## Parte 2: ¿Qué recibe cada nodo?

El grafo construye automáticamente la entrada de cada nodo:

- Los **puntos de entrada** reciben la tarea original tal cual.
- Los **nodos dependientes** reciben la tarea original **más** las salidas de sus dependencias.

Vamos a mirar el primer mensaje que recibió el `redactor`. Lo leemos del historial de su agente (`grafo.nodes["redactor"].executor`):

In [5]:
mensaje = grafo.nodes["redactor"].executor.messages[0]
print("\n".join(bloque["text"] for bloque in mensaje["content"]))

Original Task: Hola, compré unos Auriculares Inalámbricos (pedido ORD-5521) hace dos semanas y no se emparejan con mi teléfono. Ya los reinicié y nada. ¡Estoy bastante molesto!

Inputs from previous nodes:

From sentimiento:
  - Agent: **Tono:** Molesto — el cliente expresa frustración explícita ("¡Estoy bastante molesto!") pero sin llegar a ser agresivo o iracundo.
**Urgencia:** Media — el problema lleva dos semanas y ya intentó resolverlo solo, lo que aumenta la impaciencia, aunque no reporta una necesidad crítica inmediata.


From politicas:
  - Agent: **Política aplicable: Garantía** — Los auriculares presentan un fallo de funcionamiento (no emparejamiento) tras solo dos semanas de uso, lo que indica un posible defecto de fábrica cubierto por el año de garantía.



Fíjate en el formato: `Original Task: ...` seguido de `Inputs from previous nodes:` con un apartado `From <nodo>:` por cada dependencia. El redactor no necesita saber nada del grafo: le llega todo el contexto en su mensaje.

> ⚠️ **Detalle del SDK de Python:** un nodo con varias aristas de entrada se ejecuta en cuanto **cualquiera** de ellas se cumple (semántica OR). Aquí el redactor recibió las dos salidas porque `sentimiento` y `politicas` se ejecutan en el mismo lote (Python ejecuta el grafo por lotes y espera a que termine el lote completo). Si las ramas tuvieran longitudes distintas, tendrías que esperar explícitamente a todas con una condición:
>
> ```python
> from strands.multiagent.base import Status
>
> def todas_completadas(nodos: list[str]):
>     def condicion(state) -> bool:
>         return all(n in state.results and state.results[n].status == Status.COMPLETED for n in nodos)
>     return condicion
>
> builder.add_edge("sentimiento", "redactor", condition=todas_completadas(["sentimiento", "politicas"]))
> builder.add_edge("politicas", "redactor", condition=todas_completadas(["sentimiento", "politicas"]))
> ```

---

## Parte 3: Aristas condicionales — enrutar según la intención

Una **arista condicional** solo se recorre si su función `condition` devuelve `True`. La función recibe el estado del grafo (`state`), con los resultados de los nodos que ya terminaron.

```
                      ┌─[ categoria == "tecnico" ]─────▶ soporte_tecnico
┌──────────────┐      │
│ clasificador │ ─────┼─[ categoria == "facturacion" ]─▶ facturacion
└──────────────┘      │
                      └─[ plan == "premium" ]──────────▶ gestor_vip
```

Dos buenas prácticas:

1. **Salida estructurada en el clasificador.** En lugar de buscar palabras en un texto libre, el clasificador devuelve un objeto Pydantic (`structured_output_model`) y la condición lee un campo con tipo: `...result.structured_output.categoria`.
2. **Contexto en tiempo de ejecución** (`invocation_state`). Hay datos que no deben pasar por el LLM, como el plan del cliente. Se los pasas al grafo al invocarlo y llegan a las condiciones que declaran el parámetro `invocation_state` (también llegan a hooks y herramientas).

El especialista técnico usa una herramienta y lo creamos con `crear_soporte_tecnico()` de `herramientas_soporte.py`. En la Parte 6 moverás **ese mismo agente** a otro servicio.

In [6]:
from typing import Literal

from pydantic import BaseModel, Field

from herramientas_soporte import crear_soporte_tecnico


class Clasificacion(BaseModel):
    categoria: Literal["tecnico", "facturacion"] = Field(description="Tipo de ticket")
    motivo: str = Field(description="Explicación breve de la clasificación")


def crear_clasificador() -> Agent:
    return Agent(
        name="clasificador",
        system_prompt="Clasifica el ticket de soporte: 'tecnico' si es un problema con un dispositivo, "
        "'facturacion' si trata de cobros, pagos o reembolsos.",
        structured_output_model=Clasificacion,
        callback_handler=None,
    )


def crear_facturacion() -> Agent:
    return Agent(
        name="facturacion",
        system_prompt=f"Eres el equipo de facturación de TechStore. Responde al cliente en español, "
        f"en menos de 80 palabras, aplicando estas políticas:\n{POLITICAS}",
        callback_handler=None,
    )


# --- Condiciones de las aristas ---

def es_tecnico(state) -> bool:
    return state.results["clasificador"].result.structured_output.categoria == "tecnico"


def es_facturacion(state) -> bool:
    return state.results["clasificador"].result.structured_output.categoria == "facturacion"


def es_premium(state, *, invocation_state, **kwargs) -> bool:
    """Lee el plan del cliente del contexto de ejecución, no de lo que diga el LLM."""
    return invocation_state.get("plan") == "premium"


def construir_grafo_soporte():
    gestor_vip = Agent(
        name="gestor_vip",
        system_prompt="Escribe una nota interna de 2 líneas para el gestor de cuentas premium: "
        "qué le pasa al cliente y qué acción proactiva conviene tomar.",
        callback_handler=None,
    )

    builder = GraphBuilder()
    builder.add_node(crear_clasificador(), "clasificador")
    builder.add_node(crear_soporte_tecnico(), "soporte_tecnico")
    builder.add_node(crear_facturacion(), "facturacion")
    builder.add_node(gestor_vip, "gestor_vip")

    builder.add_edge("clasificador", "soporte_tecnico", condition=es_tecnico)
    builder.add_edge("clasificador", "facturacion", condition=es_facturacion)
    builder.add_edge("clasificador", "gestor_vip", condition=es_premium)

    builder.set_execution_timeout(300)
    return builder.build()


print("✅ Grafo de soporte definido")
mermaid(a_mermaid(construir_grafo_soporte()))

✅ Grafo de soporte definido


Cada flecha lleva el nombre de su función de condición. Como las tres salen del mismo nodo, se pueden activar varias a la vez.

**Caso 1:** ticket técnico de un cliente con plan básico. Solo debería ejecutarse la rama técnica.

In [7]:
resultado = construir_grafo_soporte()(TICKET_TECNICO, invocation_state={"plan": "basico"})
resumen(resultado)

print("\nClasificación:", resultado.results["clasificador"].result.structured_output)
display(Markdown(str(resultado.results["soporte_tecnico"].result)))

Estado: Status.COMPLETED
Orden de ejecución: clasificador → soporte_tecnico
Tiempo: 8.6 s | Tokens: 3445

Clasificación: categoria='tecnico' motivo='El cliente reporta un problema con sus auriculares inalámbricos (pedido ORD-5521): no logran emparejarse con su teléfono a pesar de haberlos reiniciado. Se trata de un fallo de funcionamiento en un dispositivo físico.'


¡Perfecto, encontré el problema! Aquí tienes el diagnóstico y los pasos para resolverlo:

---

## 🔍 Diagnóstico

Tus auriculares presentan un **fallo conocido de emparejamiento Bluetooth 5.0**, que puede ocurrir especialmente con ciertos modelos de teléfono. El reinicio estándar que realizaste no es suficiente para resolverlo; se necesita un **restablecimiento completo** del módulo Bluetooth.

---

## ✅ Pasos para solucionarlo

1. **Olvida el dispositivo en tu teléfono** — Ve a *Configuración > Bluetooth*, busca tus auriculares en la lista y selecciona "Olvidar" o "Eliminar".

2. **Realiza el restablecimiento completo** — Con los auriculares encendidos, **mantén pulsado el botón de encendido durante 10 segundos** hasta que el LED parpadee de forma diferente (generalmente en rojo y azul alternado). Esto borra la memoria de emparejamiento.

3. **Vuelve a emparejarlos** — Pon los auriculares en **modo de emparejamiento** (el LED debería parpadear rápidamente en azul) y búscalos desde el Bluetooth de tu teléfono.

4. **Si el problema persiste** — Asegúrate de que tu teléfono tenga la versión de Bluetooth compatible. De lo contrario, contáctanos con tu número de pedido **ORD-5521** para gestionar una revisión o reemplazo sin costo, ya que el producto tiene menos de 30 días.

---

Prueba estos pasos y cuéntame cómo te va. ¡Estoy aquí para ayudarte! 😊


**Caso 2:** ticket de facturación de un cliente **premium**. Se activan dos ramas a la vez: facturación responde al cliente y el gestor VIP recibe una nota interna.

In [8]:
resultado = construir_grafo_soporte()(TICKET_FACTURACION, invocation_state={"plan": "premium"})
resumen(resultado)

for nodo in ["facturacion", "gestor_vip"]:
    display(Markdown(f"### 🧩 `{nodo}`\n\n{resultado.results[nodo].result}"))

Estado: Status.COMPLETED
Orden de ejecución: clasificador → facturacion → gestor_vip
Tiempo: 3.8 s | Tokens: 1600


### 🧩 `facturacion`

Hola, lamentamos el inconveniente con tu pedido ORD-7788. Hemos registrado tu reporte sobre el cargo duplicado en el Teclado Mecánico. Procederemos a verificar el cobro y, una vez confirmado, realizaremos el reembolso en un plazo de **3 a 5 días hábiles**. Si tienes dudas, no dudes en contactarnos. ¡Gracias por tu paciencia! 😊


### 🧩 `gestor_vip`

# Nota Interna — Gestor de Cuentas Premium

**Situación:** El cliente ha sido cobrado dos veces por el pedido ORD-7788 (Teclado Mecánico) y solicita la devolución inmediata del cargo duplicado.
**Acción recomendada:** Verificar con facturación el doble cargo en ORD-7788, gestionar el reembolso con prioridad y contactar proactivamente al cliente para confirmar la resolución y reforzar la confianza.


---

## Parte 4: Ciclos — un bucle de revisión con límites

Los grafos pueden volver a un nodo anterior. Es la forma natural de modelar **"redacta, revisa y corrige hasta que esté bien"**:

```
┌───────────┐          ┌───────────┐
│ redactor  │ ───────▶ │  revisor  │ ──[ aprobada ]──▶ fin
└───────────┘          └───────────┘
      ▲                      │
      └──[ no aprobada ]─────┘
```

El revisor tiene una lista de comprobación que incluye una firma exacta que el redactor **no conoce**. Así el primer borrador casi seguro se rechaza y verás el ciclo en acción. Cuando el revisor aprueba, ninguna arista se cumple y el grafo termina.

Todo ciclo necesita **límites de seguridad**:

| Método | Para qué |
|---|---|
| `set_entry_point("redactor")` | Obligatorio aquí: el ciclo hace que todos los nodos tengan aristas de entrada |
| `set_max_node_executions(6)` | Máximo de ejecuciones de nodos en total (evita bucles infinitos) |
| `set_execution_timeout(300)` | Tiempo máximo del grafo completo |
| `reset_on_revisit(True)` | Cada vuelta empieza con el agente "limpio"; recibe la tarea y los comentarios del revisor |

In [9]:
class Revision(BaseModel):
    aprobada: bool = Field(description="True solo si se cumplen TODOS los criterios")
    comentarios: str = Field(description="Qué hay que corregir, o 'OK' si está aprobada")


def necesita_revision(state) -> bool:
    return not state.results["revisor"].result.structured_output.aprobada


def construir_grafo_revision():
    redactor = Agent(
        name="redactor",
        system_prompt=f"Redacta una respuesta breve en español al ticket del cliente. "
        f"Si recibes comentarios del revisor, reescribe la respuesta aplicándolos TODOS.\n\nPolíticas:\n{POLITICAS}",
        callback_handler=None,
    )
    revisor = Agent(
        name="revisor",
        system_prompt="""Revisa la respuesta al cliente con esta lista de comprobación:
1. Menciona el número de pedido.
2. Indica un plazo concreto.
3. Tiene como máximo 100 palabras.
4. Termina exactamente con la firma "— Equipo TechStore".
Apruébala solo si cumple los 4 criterios. Si no, explica qué falta.""",
        structured_output_model=Revision,
        callback_handler=None,
    )

    builder = GraphBuilder()
    builder.add_node(redactor, "redactor")
    builder.add_node(revisor, "revisor")

    builder.add_edge("redactor", "revisor")
    builder.add_edge("revisor", "redactor", condition=necesita_revision)

    builder.set_entry_point("redactor")
    builder.set_max_node_executions(6)
    builder.set_execution_timeout(300)
    builder.reset_on_revisit(True)
    return builder.build()


grafo_revision = construir_grafo_revision()
mermaid(a_mermaid(grafo_revision))

En el diagrama se ve el ciclo: `redactor → revisor` siempre, y `revisor → redactor` solo si `necesita_revision`. Lo ejecutamos:

In [10]:
resultado = grafo_revision(TICKET_FACTURACION)
resumen(resultado)

vueltas = sum(1 for nodo in resultado.execution_order if nodo.node_id == "revisor")
revision = resultado.results["revisor"].result.structured_output
print(f"\nRevisiones: {vueltas} | Aprobada: {revision.aprobada} | Comentarios: {revision.comentarios}")
display(Markdown(f"### ✉️ Respuesta final\n\n{resultado.results['redactor'].result}"))

Estado: Status.COMPLETED
Orden de ejecución: redactor → revisor → redactor → revisor
Tiempo: 9.0 s | Tokens: 4775

Revisiones: 2 | Aprobada: True | Comentarios: OK


### ✉️ Respuesta final

Hola,

Lamentamos el inconveniente con tu pedido ORD-7788. Hemos registrado el cargo duplicado por el Teclado Mecánico y procederemos a verificarlo de inmediato. Una vez confirmado, realizaremos el reembolso del cobro extra en un plazo de 3 a 5 días hábiles.

Si tienes alguna otra consulta, no dudes en contactarnos.

— Equipo TechStore


---

## Parte 5: Streaming — ver el grafo en directo

Con `stream_async` recibes eventos mientras el grafo se ejecuta: cuándo empieza y termina cada nodo, el texto que van generando los agentes y el resultado final. Es útil para interfaces en tiempo real y para depurar.

| Evento (`type`) | Cuándo llega |
|---|---|
| `multiagent_node_start` | Empieza un nodo |
| `multiagent_node_stream` | Un agente dentro del nodo genera texto o llama a herramientas |
| `multiagent_node_stop` | Termina un nodo (incluye su resultado y tiempo) |
| `multiagent_result` | Termina el grafo |

Observa que `sentimiento` y `politicas` empiezan juntos: es el paralelismo de la Parte 1.

In [11]:
grafo = construir_grafo_respuesta()

async for evento in grafo.stream_async(TICKET_FACTURACION):
    tipo = evento.get("type")
    if tipo == "multiagent_node_start":
        print(f"▶️  {evento['node_id']} empieza")
    elif tipo == "multiagent_node_stop":
        print(f"⏹️  {evento['node_id']} termina en {evento['node_result'].execution_time / 1000:.1f} s")
    elif tipo == "multiagent_result":
        print(f"🏁 Grafo terminado: {evento['result'].status}")

▶️  recepcion empieza
⏹️  recepcion termina en 1.2 s
▶️  sentimiento empieza
▶️  politicas empieza
⏹️  politicas termina en 1.3 s
⏹️  sentimiento termina en 1.6 s
▶️  redactor empieza
⏹️  redactor termina en 3.2 s
🏁 Grafo terminado: Status.COMPLETED


---

## Parte 6: A2A — agentes que hablan entre servicios

Hasta ahora todos los nodos vivían en este notebook. En una empresa real, cada equipo suele desarrollar, desplegar y escalar **su propio agente** como un servicio: el equipo de soporte técnico mantiene su especialista y sus herramientas, y otros equipos lo consumen.

**A2A (Agent-to-Agent)** es un protocolo abierto para que agentes de distintas plataformas y frameworks se **descubran**, se **comuniquen** y **colaboren**:

| Concepto A2A | Qué es |
|---|---|
| **Agent Card** | Una ficha JSON en `/.well-known/agent-card.json` con el nombre, la descripción y las habilidades (*skills*) del agente. Sirve para descubrirlo |
| **Mensajes** | Peticiones JSON-RPC sobre HTTP en la ruta raíz del servidor, con soporte de streaming |
| **`context_id`** | Identifica cada conversación; el servidor aísla el historial de cada una |

Strands soporta A2A en los dos lados:

- **Servidor:** `A2AServer` expone un agente de Strands por A2A.
- **Cliente:** `A2AAgent` consume un agente remoto **como si fuera local**. Devuelve el mismo `AgentResult`, así que encaja como nodo de un Graph o dentro de una herramienta.

```
   Este notebook (orquestación)                          Servicio de soporte técnico
┌──────────────────────────────────┐   HTTP + A2A    ┌───────────────────────────────┐
│ clasificador ─▶ A2AAgent ────────┼───────────────▶ │ A2AServer                     │
│      │          (nodo remoto)    │ ◀───────────────┼─  └─ soporte_tecnico + tool   │
│      └──────▶ facturacion        │                 └───────────────────────────────┘
└──────────────────────────────────┘
```

### 6.1 El servidor

El archivo `servidor_soporte_tecnico.py` expone al especialista técnico. Lo esencial son tres líneas:

```python
from strands.multiagent.a2a import A2AServer
from herramientas_soporte import crear_soporte_tecnico

A2AServer(agent_factory=crear_soporte_tecnico, host="127.0.0.1", port=puerto).serve()
```

`agent_factory` es la opción recomendada: el servidor llama a la fábrica una vez **por conversación** (`context_id`). Así, cada cliente tiene su propio agente, sus historiales no se mezclan y varias conversaciones pueden atenderse a la vez.

En producción este servidor sería **otro proceso, contenedor o máquina**. Aquí lo lanzamos en segundo plano desde el notebook. Pedimos al sistema un **puerto libre** para no chocar con otros servicios (por ejemplo, el propio kernel de Jupyter) y esperamos a que publique su Agent Card. Sus logs se guardan en `servidor_a2a.log`.

In [12]:
import socket
import subprocess
import sys
import time

import httpx


def puerto_libre() -> int:
    """Pide al sistema operativo un puerto TCP libre."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


PUERTO = puerto_libre()
URL_SOPORTE = f"http://127.0.0.1:{PUERTO}"

log_servidor = open("servidor_a2a.log", "w")
servidor = subprocess.Popen(
    [sys.executable, "servidor_soporte_tecnico.py", str(PUERTO)],
    stdout=log_servidor,
    stderr=subprocess.STDOUT,
)

# Esperamos a que el servidor publique su Agent Card
for _ in range(30):
    if servidor.poll() is not None:
        raise RuntimeError("El servidor A2A se detuvo. Revisa servidor_a2a.log")
    try:
        httpx.get(f"{URL_SOPORTE}/.well-known/agent-card.json").raise_for_status()
        break
    except httpx.HTTPError:
        time.sleep(1)
else:
    raise RuntimeError("El servidor A2A no respondió a tiempo. Revisa servidor_a2a.log")

print(f"✅ Servidor A2A escuchando en {URL_SOPORTE}")

✅ Servidor A2A escuchando en http://127.0.0.1:55827


### 6.2 El cliente: `A2AAgent`

Solo necesitas la URL. `A2AAgent` descarga la Agent Card, prepara el cliente HTTP y construye los mensajes del protocolo por ti. Primero miramos la ficha del agente remoto. Fíjate en que las *skills* se generan solas a partir de sus herramientas:

In [13]:
from strands.agent.a2a_agent import A2AAgent

soporte_remoto = A2AAgent(endpoint=URL_SOPORTE, timeout=120)

tarjeta = await soporte_remoto.get_agent_card()
print(f"Nombre:      {tarjeta.name}")
print(f"Descripción: {tarjeta.description}")
print(f"URL:         {tarjeta.url}")
print(f"Skills:      {[skill.name for skill in tarjeta.skills]}")

Nombre:      soporte_tecnico
Descripción: Especialista de TechStore que diagnostica problemas de dispositivos electrónicos y propone soluciones paso a paso.
URL:         http://127.0.0.1:55827/
Skills:      ['buscar_problemas_conocidos']


Ahora lo invocamos **igual que a un agente local**. Devuelve un `AgentResult`, el mismo tipo que un `Agent` normal. La herramienta `buscar_problemas_conocidos` se ejecuta **en el servidor**, no aquí.

In [14]:
respuesta = await soporte_remoto.invoke_async("Mi Hub USB-C no muestra imagen en el monitor externo.")
print("Tipo de resultado:", type(respuesta).__name__)
display(Markdown(str(respuesta)))

Tipo de resultado: AgentResult


¡Gracias por la información! Aquí tienes el diagnóstico y los pasos para resolver el problema:

---

### 🔍 Diagnóstico
Tu portátil podría no tener habilitado el **modo alternativo DisplayPort** en el puerto USB-C que estás usando, lo que impide que el Hub transmita señal de vídeo al monitor externo.

---

### ✅ Pasos para resolverlo

1. **Verifica las especificaciones de tu portátil:** Consulta el manual o la página del fabricante y asegúrate de que el puerto USB-C que usas soporta **modo alternativo DisplayPort (DP Alt Mode)**. No todos los puertos USB-C tienen esta capacidad.

2. **Prueba con un puerto Thunderbolt (⚡):** Si tu portátil tiene un puerto con el icono de rayo, conéctalo ahí. Los puertos Thunderbolt 3 y 4 son totalmente compatibles con salida de vídeo.

3. **Comprueba el cable y el monitor:** Asegúrate de que el cable entre el Hub y el monitor es compatible (HDMI, DisplayPort, etc.) y que el monitor está en la **fuente de entrada correcta**.

4. **Actualiza los controladores:** Ve al administrador de dispositivos de tu sistema operativo y actualiza los drivers del adaptador de pantalla y del puerto USB-C/Thunderbolt.

---

Si tras estos pasos el problema persiste, tráenos el dispositivo a la tienda y lo revisamos en detalle. 😊



### 6.3 `A2AAgent` como nodo de un Graph

Esta es la integración clave: un `A2AAgent` se añade al grafo **exactamente igual que un agente local**. Reutilizamos el grafo de la Parte 3 cambiando un solo nodo: `soporte_tecnico` ya no es un agente local, es el servicio remoto.

La orquestación (clasificar y decidir la rama) sigue ocurriendo aquí; el trabajo especializado ocurre en el otro servicio. Usamos `invoke_async`, la forma recomendada en la documentación para grafos con nodos remotos.

In [15]:
builder = GraphBuilder()
builder.add_node(crear_clasificador(), "clasificador")
builder.add_node(soporte_remoto, "soporte_tecnico")  # ← nodo remoto por A2A
builder.add_node(crear_facturacion(), "facturacion")

builder.add_edge("clasificador", "soporte_tecnico", condition=es_tecnico)
builder.add_edge("clasificador", "facturacion", condition=es_facturacion)
builder.set_execution_timeout(300)
grafo_distribuido = builder.build()
mermaid(a_mermaid(grafo_distribuido))

El nodo en ámbar es el remoto: para el grafo es un nodo más, aunque detrás haya una llamada HTTP a otro servicio. Lo ejecutamos:

In [16]:
resultado = await grafo_distribuido.invoke_async(TICKET_TECNICO)
resumen(resultado)
display(Markdown(f"### 🌐 Respuesta del nodo remoto\n\n{resultado.results['soporte_tecnico'].result}"))

Estado: Status.COMPLETED
Orden de ejecución: clasificador → soporte_tecnico
Tiempo: 9.4 s | Tokens: 960


### 🌐 Respuesta del nodo remoto

¡Buenas noticias! Encontré el problema. 🎯

---

## 🔍 Diagnóstico

Tus auriculares están experimentando un **fallo de emparejamiento Bluetooth 5.0**, que puede ocurrir especialmente con ciertos modelos de teléfonos. El reinicio estándar que realizaste no es suficiente en este caso; se necesita un **restablecimiento completo** del dispositivo.

---

## ✅ Pasos para solucionarlo

1. **🔴 Restablece los auriculares por completo:** Mantén pulsado el **botón de encendido durante 10 segundos** hasta que veas un destello de luz o escuches un sonido de confirmación. Esto es diferente a un reinicio normal.

2. **📱 Elimina el emparejamiento anterior en tu teléfono:** Ve a *Ajustes > Bluetooth* y borra los auriculares de la lista de dispositivos vinculados.

3. **🔗 Vuelve a emparejar desde cero:** Con los auriculares en modo de emparejamiento (luz parpadeante), búscalos nuevamente desde el Bluetooth de tu teléfono y conéctalos.

4. **📲 Verifica la compatibilidad Bluetooth:** Si el problema persiste, comprueba en los ajustes de tu teléfono que tenga **Bluetooth 5.0 o superior**, ya que puede haber incompatibilidad con versiones anteriores.

---

Si sigues estos pasos y el problema continúa, no dudes en contactarnos de nuevo con tu pedido **ORD-5521** y gestionaremos una solución adicional (cambio o revisión técnica). 🙌 ¡Mucho ánimo, lo vamos a solucionar!



¿Cómo sabemos que la respuesta vino del otro servicio? Por su log: cada `POST /` es un mensaje A2A recibido, y cada `GET /.well-known/agent-card.json` es una consulta de la Agent Card.

Fíjate también en los tokens del resumen: son muchos menos que en la Parte 3, porque el consumo del nodo remoto ocurre (y se mide) en el otro servicio.

In [17]:
log_servidor.flush()
for linea in open("servidor_a2a.log"):
    if "agent-card.json" in linea or '"POST / ' in linea:
        print(linea.rstrip())

INFO:     127.0.0.1:40080 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:40090 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:40094 - "POST / HTTP/1.1" 200 OK
INFO:     127.0.0.1:47540 - "POST / HTTP/1.1" 200 OK


### 6.4 `A2AAgent` como herramienta

El otro patrón agente a agente es el del Módulo 6 (*agents as tools*), pero con un especialista remoto: lo envuelves en una herramienta y es el orquestador quien decide cuándo llamarlo.

In [18]:
from strands import tool


@tool
def consultar_soporte_tecnico(problema: str) -> str:
    """Consulta al especialista de soporte técnico (servicio remoto A2A) sobre un problema con un dispositivo.

    Args:
        problema: Descripción del problema, incluido el nombre del dispositivo
    """
    return str(soporte_remoto(problema))


orquestador = Agent(
    system_prompt="Eres el asistente de TechStore. Para problemas técnicos con dispositivos usa "
    "consultar_soporte_tecnico y resume la solución al cliente en un tono cercano.",
    tools=[consultar_soporte_tecnico],
    callback_handler=None,
)

display(Markdown(str(orquestador("Mi teclado mecánico escribe algunas letras dos veces. ¿Qué hago?"))))

¡Buenas noticias! 🎉 Nuestro equipo técnico ya identificó el problema. Aquí te explico qué está pasando y cómo solucionarlo:

---

**¿Qué ocurre?** Tu teclado está registrando dobles pulsaciones con una sola presión de tecla. Esto se debe a un **bug en el firmware v2.1**.

**¿Cómo lo arreglas?** ¡Con una simple actualización! Sigue estos pasos:

1. 🔍 **Comprueba tu versión de firmware** desde la aplicación del fabricante o el manual de tu teclado.
2. 💾 **Descarga el firmware v2.3** desde el sitio web oficial del fabricante (sección de descargas o soporte).
3. ⚡ **Instala la actualización** conectando el teclado por USB y siguiendo el asistente de instalación.
4. ✅ **Prueba el teclado** abriendo un bloc de notas y presionando las teclas que fallaban.

---

💡 **Tip extra:** Si tras actualizar el problema continúa, es posible que haya desgaste físico en los switches de esas teclas. En ese caso, ¡pásate por nuestra tienda y te echamos un vistazo! 🛠️

¿Te puedo ayudar con algo más? 😊


### 6.5 Hacia producción: buenas prácticas de A2A

| Tema | Recomendación de la documentación |
|---|---|
| Aislamiento | Usa `agent_factory` (un agente por `context_id`). El servidor guarda como máximo `max_contexts` conversaciones (1000 por defecto) y descarta las menos usadas |
| Autenticación | El `context_id` **no** es un mecanismo de seguridad: quien lo conozca puede unirse a esa conversación. Autentica en la capa de transporte o en un gateway |
| Cliente autenticado | Pasa a `A2AAgent` un `client_config` con un `httpx_client` configurado (SigV4, OAuth o *bearer tokens*) |
| Red | Por defecto el servidor escucha solo en `127.0.0.1`. Detrás de un balanceador, indica la URL pública con `http_url` |
| Tiempos | `A2AAgent` espera 300 s por defecto (`timeout`). Ajústalo a lo que tarde el agente remoto |
| Madurez | La integración A2A de Strands es **experimental** (lo verás en el log del servidor). Revisa los cambios al actualizar |

### 6.6 Apagar el servidor

In [19]:
servidor.terminate()
servidor.wait()
log_servidor.close()
print("🛑 Servidor A2A detenido")

🛑 Servidor A2A detenido


---

## Más allá: nodos anidados y nodos personalizados

- **Grafos anidados:** un nodo puede ser otro `Graph` o un `Swarm`. Por ejemplo, un enjambre de investigadores como un solo nodo dentro de un proceso más grande:

  ```python
  from strands.multiagent import Swarm

  equipo = Swarm([investigador_tecnico, investigador_mercado])
  builder.add_node(equipo, "equipo_investigacion")
  builder.add_edge("equipo_investigacion", "analista")
  ```

- **Nodos personalizados:** si un paso es lógica de negocio determinista (validar datos, aplicar una regla), puedes heredar de `MultiAgentBase` y crear un nodo que no llama al LLM.

- **Otros detalles del SDK de Python:**
  - Los fallos de un nodo lanzan una excepción (*fail-fast*); superar un límite del grafo devuelve estado `FAILED`.
  - Un hook puede cancelar un nodo antes de ejecutarse; en Python el nodo cancelado queda como `FAILED`.

---

## Resumen

| Lo que hiciste | Pieza de Strands |
|---|---|
| Definiste nodos, aristas y un punto de entrada | `GraphBuilder.add_node`, `add_edge`, `set_entry_point` |
| Ejecutaste ramas en paralelo y las agregaste | Topología en diamante + propagación de entrada |
| Enrutaste tickets según su intención y el plan del cliente | Aristas condicionales + `structured_output_model` + `invocation_state` |
| Creaste un bucle de revisión seguro | Ciclos + `set_max_node_executions`, `set_execution_timeout`, `reset_on_revisit` |
| Observaste la ejecución en directo | `stream_async` |
| Dibujaste cada grafo a partir del objeto real | `grafo.edges`, `grafo.nodes` + Mermaid |
| Expusiste un agente como servicio | `A2AServer(agent_factory=...)` |
| Consumiste un agente remoto como nodo y como herramienta | `A2AAgent` |

**La idea central:** el Graph te da un proceso **predecible** (tú decides el camino), los agentes aportan el razonamiento en cada paso, y A2A te permite repartir esos pasos entre **servicios independientes** sin cambiar la forma del grafo.

---

## 🎯 Pruébalo tú

1. **Nueva rama.** Añade la categoría `"devolucion"` a `Clasificacion` y un nodo `devoluciones` con su arista condicional. Pruébalo con: *"Quiero devolver el Hub USB-C, no lo he usado"*.
2. **Menos vueltas.** Añade la firma `— Equipo TechStore` al `system_prompt` del redactor de la Parte 4. ¿Cuántas revisiones hacen falta ahora?
3. **Espera explícita.** Usa la condición `todas_completadas` de la Parte 2 en el grafo de la Parte 1 y comprueba que el resultado es el mismo.
4. **Segundo servicio A2A.** Crea `servidor_facturacion.py` con el agente de facturación, lánzalo en otro puerto y sustituye también el nodo `facturacion` por un `A2AAgent`.

## 📚 Para saber más

- [Graph](https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph/) y [Graph components](https://strandsagents.com/docs/user-guide/sdk/multi-agent/graph-components/)
- [Agent-to-Agent (A2A)](https://strandsagents.com/docs/user-guide/sdk/multi-agent/agent-to-agent/) y [A2A Server Configuration](https://strandsagents.com/docs/user-guide/sdk/multi-agent/a2a-server-configuration/)
- [Coordinate multiple agents](https://strandsagents.com/docs/user-guide/sdk/multi-agent/multi-agent-patterns/) (Graph vs Swarm vs Workflow)
- [Protocolo A2A](https://a2aproject.github.io/A2A/latest/)